# Google Drive에서 CSV 파일 로드 및 웹 앱 준비

먼저 Google Drive를 마운트하여 Colab에서 파일에 접근할 수 있도록 합니다. 실행하면 Google 계정 인증 요청이 나타날 것입니다.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Google Drive가 성공적으로 마운트되면, `new` 폴더 안에 있는 `web.csv` 파일을 로드하겠습니다. `encoding='utf-8-sig'`를 사용하여 한국어 문자가 올바르게 표시되도록 합니다.

In [4]:
import pandas as pd

file_path = '/content/drive/MyDrive/new/web.csv'

try:
    df = pd.read_csv(file_path, encoding='cp949') # Changed encoding to cp949
    print(f"'{file_path}' 파일이 성공적으로 로드되었습니다.")
    print("데이터의 처음 5행을 표시합니다:")
    display(df.head())
except FileNotFoundError:
    print(f"오류: '{file_path}' 파일을 찾을 수 없습니다. 경로가 올바른지, 파일이 Google Drive의 'new' 폴더에 있는지 확인해 주세요.")
except UnicodeDecodeError:
    print(f"오류: '{file_path}' 파일의 인코딩을 'cp949'로 디코딩할 수 없습니다. 다른 인코딩(예: 'euc-kr', 'latin1')을 시도해 볼 수 있습니다.")
except Exception as e:
    print(f"파일 로드 중 오류가 발생했습니다: {e}")

'/content/drive/MyDrive/new/web.csv' 파일이 성공적으로 로드되었습니다.
데이터의 처음 5행을 표시합니다:


,1,서울역1~22,1호선 1번 출입구 주변
0,1,서울역23~44,"2,3번 출입구 복도 사이"
1,1,서울역45~64,"2,3번 출입구 복도 사이/화장실 주변"
2,1,시청29~44,1번 출구 주변
3,1,시청1~28,"2,3번 출구 주변"
4,1,종각1~25,4번 출구 주변


In [5]:
!pip install -q gradio

In [8]:
import pandas as pd
import gradio as gr
import re
from google.colab import drive

drive.mount('/content/drive')

# ── 1. 데이터 로드 ──────────────────────────────
CSV_PATH = "/content/drive/MyDrive/new/web.csv"   # 헤더 없음 → header=None 필수

df = pd.read_csv(
    CSV_PATH,
    encoding="cp949",
    header=None,
    names=["호선", "보관함명", "상세위치"],
)
df["호선"] = df["호선"].astype(str)

line_options = ["전체"] + sorted(
    df["호선"].unique(), key=lambda x: (not x.isdigit(), x)
)

# ── 2. 'n번' 볼드 처리 ──────────────────────────
def bold_exits(text):
    return re.sub(r'(\d[\d,\s\-~]*번)', r'**\1**', str(text))

def style_table(frame):
    disp = frame.copy()
    disp["상세위치"] = disp["상세위치"].apply(bold_exits)
    return disp

# 결과가 없을 때 보여줄 표 (상세위치 칸에 안내 문구)
EMPTY_TABLE = pd.DataFrame([["-", "-", "**물품보관함 없음**"]],
                           columns=["호선", "보관함명", "상세위치"])

# ── 3. 검색 함수 ────────────────────────────────
def search(line, keyword):
    out = df  # 검색은 항상 원본 텍스트로
    if line and line != "전체":
        out = out[out["호선"] == line]
    if keyword and keyword.strip():
        k = keyword.strip()
        mask = out["보관함명"].str.contains(k, case=False, na=False) | \
               out["상세위치"].str.contains(k, case=False, na=False)
        out = out[mask]
    out = out.reset_index(drop=True)

    # 🔸 결과가 0건이면 '물품보관함 없음' 표시
    if len(out) == 0:
        msg = f"🚫 '{keyword.strip()}' — 물품보관함 없음" if keyword and keyword.strip() else "🚫 물품보관함 없음"
        return EMPTY_TABLE, msg

    return style_table(out), f"🔎 검색 결과: 총 {len(out)}건"

# ── 4. UI 구성 ──────────────────────────────────
with gr.Blocks(title="지하철 물품보관함 찾기") as demo:
    gr.Markdown("## 🚇 지하철 물품보관함 찾기\n호선을 고르거나 역/위치 이름을 입력해 보세요. **출구 번호**는 굵게 표시됩니다.")

    with gr.Row():
        line_dd = gr.Dropdown(line_options, value="전체", label="호선", scale=1)
        kw_tb = gr.Textbox(label="검색어 (역명·위치)", placeholder="예: 서울역, 강남, 출구", scale=2)

    count_md = gr.Markdown(f"🔎 검색 결과: 총 {len(df)}건")
    table = gr.Dataframe(
        value=style_table(df),
        headers=["호선", "보관함명", "상세위치"],
        datatype=["str", "str", "markdown"],
        wrap=True,
        interactive=False,
    )

    line_dd.change(search, [line_dd, kw_tb], [table, count_md])
    kw_tb.change(search, [line_dd, kw_tb], [table, count_md])

# ── 5. 실행 ────────────────────────────────────
demo.launch(share=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://de8bc82f55c2d7884d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.io/spaces)
